## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = "Data/Laghouat_2022_final.csv"
OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

print("Setup complete")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Setup complete
Pandas: 2.2.3
NumPy: 2.2.3


## Load data

In [3]:
df_raw = pd.read_csv(DATA_PATH)

df_raw["datetime"] = pd.to_datetime(
    df_raw["datetime"],
    format="%d/%m/%Y %H:%M"
)

df_raw = df_raw.sort_values("datetime").reset_index(drop=True)
df_raw = df_raw.set_index("datetime")

print("Raw shape:", df_raw.shape)
print("Date range:", df_raw.index.min(), "→", df_raw.index.max())
df_raw.head()

Raw shape: (4368, 13)
Date range: 2022-02-17 00:00:00 → 2022-08-17 23:00:00


,dust_density_measured_gm2,dust_density_reconstructed_PCHIP_gm2,temp_amb_C,T_cell_C,G_POA_Wm2,RH_pct,wind_speed_ms,cloud_cover_pct,precipitation_mm,P_clean_W,P_dusty_W,P_clean_norm,P_dusty_norm
datetime,,,,,,,,,,,,,
2022-02-17 00:00:00,NaN,0.000,9.000,9.000,0.000,26,4.700,1,0.000,0.000,0.000,0.000,0.000
2022-02-17 01:00:00,NaN,0.000,8.100,8.100,0.000,33,4.500,0,0.000,0.000,0.000,0.000,0.000
2022-02-17 02:00:00,NaN,0.000,7.100,7.100,0.000,37,3.600,29,0.000,0.000,0.000,0.000,0.000
2022-02-17 03:00:00,NaN,0.000,5.500,5.500,0.000,44,2.600,31,0.000,0.000,0.000,0.000,0.000
2022-02-17 04:00:00,NaN,0.000,4.600,4.600,0.000,49,2.400,15,0.000,0.000,0.000,0.000,0.000


In [4]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean() * 100,
    "unique": df_raw.nunique()
})

audit

,dtype,missing,missing_pct,unique
dust_density_measured_gm2,float64,4320,98.901,48
dust_density_reconstructed_PCHIP_gm2,float64,0,0.000,4202
temp_amb_C,float64,0,0.000,404
T_cell_C,float64,0,0.000,688
G_POA_Wm2,float64,0,0.000,2305
RH_pct,int64,0,0.000,94
wind_speed_ms,float64,0,0.000,137
cloud_cover_pct,int64,0,0.000,101
precipitation_mm,float64,0,0.000,22
P_clean_W,float64,0,0.000,1770


In [5]:
# Drop unusable or redundant columns
df = df_raw.copy()

drop_cols = [
    "dust_density_measured_gm2",  # 98.9% missing
    "P_clean_norm",               # normalized version
    "P_dusty_norm"                # normalized version
]

df = df.drop(columns=drop_cols)

print("Shape after dropping columns:", df.shape)
df.columns.tolist()

Shape after dropping columns: (4368, 10)


['dust_density_reconstructed_PCHIP_gm2',
 'temp_amb_C',
 'T_cell_C',
 'G_POA_Wm2',
 'RH_pct',
 'wind_speed_ms',
 'cloud_cover_pct',
 'precipitation_mm',
 'P_clean_W',
 'P_dusty_W']

##  Create soiling loss features

In [6]:
df["soiling_loss_W"] = df["P_clean_W"] - df["P_dusty_W"]

df["soiling_loss_pct"] = np.where(
    df["P_clean_W"] > 0,
    (df["soiling_loss_W"] / df["P_clean_W"]) * 100,
    0
)

df[["P_clean_W", "P_dusty_W", "soiling_loss_W", "soiling_loss_pct"]].head(10)

,P_clean_W,P_dusty_W,soiling_loss_W,soiling_loss_pct
datetime,,,,
2022-02-17 00:00:00,0.000,0.000,0.000,0.000
2022-02-17 01:00:00,0.000,0.000,0.000,0.000
2022-02-17 02:00:00,0.000,0.000,0.000,0.000
2022-02-17 03:00:00,0.000,0.000,0.000,0.000
2022-02-17 04:00:00,0.000,0.000,0.000,0.000
2022-02-17 05:00:00,0.000,0.000,0.000,0.000
2022-02-17 06:00:00,0.000,0.000,0.000,0.000
2022-02-17 07:00:00,0.000,0.000,0.000,0.000
2022-02-17 08:00:00,0.770,0.450,0.320,41.558


In [7]:
# Validate daytime soiling loss 

day_df = df[df["G_POA_Wm2"] > 0]

day_summary = {
    "daytime_records": len(day_df),
    "mean_clean_power_W": day_df["P_clean_W"].mean(),
    "mean_dusty_power_W": day_df["P_dusty_W"].mean(),
    "mean_soiling_loss_W": day_df["soiling_loss_W"].mean(),
    "mean_soiling_loss_pct": day_df["soiling_loss_pct"].mean()
}

pd.Series(day_summary)

daytime_records         2367.000
mean_clean_power_W        29.936
mean_dusty_power_W        25.450
mean_soiling_loss_W        4.486
mean_soiling_loss_pct     13.069
dtype: float64

## Create calendar and cyclic time features

In [8]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["dayofyear"] = df.index.dayofyear

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)

df["is_daytime"] = (df["G_POA_Wm2"] > 0).astype(int)

df[["hour", "hour_sin", "hour_cos", "dayofyear", "doy_sin", "doy_cos", "is_daytime"]].head()

,hour,hour_sin,hour_cos,dayofyear,doy_sin,doy_cos,is_daytime
datetime,,,,,,,
2022-02-17 00:00:00,0,0.000,1.000,48,0.735,0.678,0
2022-02-17 01:00:00,1,0.259,0.966,48,0.735,0.678,0
2022-02-17 02:00:00,2,0.500,0.866,48,0.735,0.678,0
2022-02-17 03:00:00,3,0.707,0.707,48,0.735,0.678,0
2022-02-17 04:00:00,4,0.866,0.500,48,0.735,0.678,0


In [9]:
# Create lag features from past production only

target = "P_dusty_W"

lag_hours = [1, 2, 3, 6, 12, 24, 48, 72]

for lag in lag_hours:
    df[f"P_dusty_lag_{lag}h"] = df[target].shift(lag)

df[[target] + [f"P_dusty_lag_{lag}h" for lag in lag_hours]].head(30)

,P_dusty_W,P_dusty_lag_1h,P_dusty_lag_2h,P_dusty_lag_3h,P_dusty_lag_6h,P_dusty_lag_12h,P_dusty_lag_24h,P_dusty_lag_48h,P_dusty_lag_72h
datetime,,,,,,,,,
2022-02-17 00:00:00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 01:00:00,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 02:00:00,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 03:00:00,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN
2022-02-17 04:00:00,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN
2022-02-17 05:00:00,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN
2022-02-17 06:00:00,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN
2022-02-17 07:00:00,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN
2022-02-17 08:00:00,0.450,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN


In [10]:
# Create rolling statistics from past production only

windows = [3, 6, 12, 24]

for window in windows:
    df[f"P_dusty_roll_mean_{window}h"] = (
        df[target]
        .shift(1)
        .rolling(window=window)
        .mean()
    )

    df[f"P_dusty_roll_max_{window}h"] = (
        df[target]
        .shift(1)
        .rolling(window=window)
        .max()
    )

rolling_cols = [col for col in df.columns if "roll_" in col]
df[[target] + rolling_cols].head(30)

,P_dusty_W,P_dusty_roll_mean_3h,P_dusty_roll_max_3h,P_dusty_roll_mean_6h,P_dusty_roll_max_6h,P_dusty_roll_mean_12h,P_dusty_roll_max_12h,P_dusty_roll_mean_24h,P_dusty_roll_max_24h
datetime,,,,,,,,,
2022-02-17 00:00:00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 01:00:00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 02:00:00,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 03:00:00,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 04:00:00,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 05:00:00,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN
2022-02-17 06:00:00,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN
2022-02-17 07:00:00,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN
2022-02-17 08:00:00,0.450,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN


## Create dust-related features

In [11]:
df["dust_lag_24h"] = df["dust_density_reconstructed_PCHIP_gm2"].shift(24)

df["dust_roll_mean_24h"] = (
    df["dust_density_reconstructed_PCHIP_gm2"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["dust_roll_max_24h"] = (
    df["dust_density_reconstructed_PCHIP_gm2"]
    .shift(1)
    .rolling(window=24)
    .max()
)

df[[
    "dust_density_reconstructed_PCHIP_gm2",
    "dust_lag_24h",
    "dust_roll_mean_24h",
    "dust_roll_max_24h"
]].head(30)

,dust_density_reconstructed_PCHIP_gm2,dust_lag_24h,dust_roll_mean_24h,dust_roll_max_24h
datetime,,,,
2022-02-17 00:00:00,0.000,NaN,NaN,NaN
2022-02-17 01:00:00,0.000,NaN,NaN,NaN
2022-02-17 02:00:00,0.000,NaN,NaN,NaN
2022-02-17 03:00:00,0.000,NaN,NaN,NaN
2022-02-17 04:00:00,0.000,NaN,NaN,NaN
2022-02-17 05:00:00,0.000,NaN,NaN,NaN
2022-02-17 06:00:00,0.000,NaN,NaN,NaN
2022-02-17 07:00:00,0.000,NaN,NaN,NaN
2022-02-17 08:00:00,0.000,NaN,NaN,NaN


In [13]:
# Create forecast targets for 24h and 48h horizons

df["target_P_dusty_24h"] = df[target].shift(-24)
df["target_P_dusty_48h"] = df[target].shift(-48)

df[[target, "target_P_dusty_24h", "target_P_dusty_48h"]].head(30)

,P_dusty_W,target_P_dusty_24h,target_P_dusty_48h
datetime,,,
2022-02-17 00:00:00,0.000,0.000,0.000
2022-02-17 01:00:00,0.000,0.000,0.000
2022-02-17 02:00:00,0.000,0.000,0.000
2022-02-17 03:00:00,0.000,0.000,0.000
2022-02-17 04:00:00,0.000,0.000,0.000
2022-02-17 05:00:00,0.000,0.000,0.000
2022-02-17 06:00:00,0.000,0.000,0.000
2022-02-17 07:00:00,0.000,0.000,0.000
2022-02-17 08:00:00,0.450,0.800,0.830


In [16]:
# Define model features

features = [
    # Calendar features
    "hour", "dayofweek", "month", "dayofyear",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
    "is_daytime",

    # Past production
    "P_dusty_lag_1h", "P_dusty_lag_2h", "P_dusty_lag_3h",
    "P_dusty_lag_6h", "P_dusty_lag_12h", "P_dusty_lag_24h",
    "P_dusty_lag_48h", "P_dusty_lag_72h",
    "P_dusty_roll_mean_3h", "P_dusty_roll_mean_6h",
    "P_dusty_roll_mean_12h", "P_dusty_roll_mean_24h",
    "P_dusty_roll_max_3h", "P_dusty_roll_max_6h",
    "P_dusty_roll_max_12h", "P_dusty_roll_max_24h",

    # Dust features
    "dust_density_reconstructed_PCHIP_gm2",
    "dust_lag_24h",
    "dust_roll_mean_24h",
    "dust_roll_max_24h",

    # Weather features available at forecast time
    "temp_amb_C",
    "T_cell_C",
    "RH_pct",
    "wind_speed_ms",
    "cloud_cover_pct",
    "precipitation_mm",

    # Target-related helper
    "soiling_loss_W",
    "soiling_loss_pct"
]

missing_features = [col for col in features if col not in df.columns]
print("Missing features:", missing_features)
print("Number of features:", len(features))


Missing features: []
Number of features: 37


In [17]:
# Prepare datasets for 24h and 48h forecasting

df_24 = df.dropna(subset=["target_P_dusty_24h"] + features).copy()
df_48 = df.dropna(subset=["target_P_dusty_48h"] + features).copy()

print("24h dataset:", df_24.shape)
print("48h dataset:", df_48.shape)

df_24[features + ["target_P_dusty_24h"]].head()

24h dataset: (4272, 42)
48h dataset: (4248, 42)


,hour,dayofweek,month,dayofyear,hour_sin,hour_cos,doy_sin,doy_cos,is_daytime,P_dusty_lag_1h,P_dusty_lag_2h,P_dusty_lag_3h,P_dusty_lag_6h,P_dusty_lag_12h,P_dusty_lag_24h,P_dusty_lag_48h,P_dusty_lag_72h,P_dusty_roll_mean_3h,P_dusty_roll_mean_6h,P_dusty_roll_mean_12h,P_dusty_roll_mean_24h,P_dusty_roll_max_3h,P_dusty_roll_max_6h,P_dusty_roll_max_12h,P_dusty_roll_max_24h,dust_density_reconstructed_PCHIP_gm2,dust_lag_24h,dust_roll_mean_24h,dust_roll_max_24h,temp_amb_C,T_cell_C,RH_pct,wind_speed_ms,cloud_cover_pct,precipitation_mm,soiling_loss_W,soiling_loss_pct,target_P_dusty_24h
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-02-20 00:00:00,0,6,2,51,0.000,1.000,0.769,0.639,0,0.000,0.000,0.000,1.300,39.430,0.000,0.000,0.000,0.000,0.217,8.925,7.860,0.000,1.300,39.430,69.630,0.044,0.015,0.027,0.042,13.800,13.800,38,2.900,100,0.000,0.000,0.000,0.000
2022-02-20 01:00:00,1,6,2,51,0.259,0.966,0.769,0.639,0,0.000,0.000,0.000,0.000,24.960,0.000,0.000,0.000,0.000,0.000,5.639,7.860,0.000,0.000,24.960,69.630,0.045,0.016,0.029,0.044,13.500,13.500,43,4.400,100,0.000,0.000,0.000,0.000
2022-02-20 02:00:00,2,6,2,51,0.500,0.866,0.769,0.639,0,0.000,0.000,0.000,0.000,14.450,0.000,0.000,0.000,0.000,0.000,3.559,7.860,0.000,0.000,14.450,69.630,0.047,0.017,0.030,0.045,12.400,12.400,61,4.900,76,0.000,0.000,0.000,0.000
2022-02-20 03:00:00,3,6,2,51,0.707,0.707,0.769,0.639,0,0.000,0.000,0.000,0.000,12.430,0.000,0.000,0.000,0.000,0.000,2.355,7.860,0.000,0.000,12.430,69.630,0.048,0.018,0.031,0.047,11.100,11.100,71,4.600,90,0.000,0.000,0.000,0.000
2022-02-20 04:00:00,4,6,2,51,0.866,0.500,0.769,0.639,0,0.000,0.000,0.000,0.000,9.470,0.000,0.000,0.000,0.000,0.000,1.319,7.860,0.000,0.000,9.470,69.630,0.050,0.019,0.032,0.048,10.400,10.400,76,3.600,92,0.000,0.000,0.000,0.000


In [18]:
#  Create chronological train/validation/test splits

def chronological_split(data, target_col):
    n = len(data)

    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    train_df = data.iloc[:train_end]
    val_df = data.iloc[train_end:val_end]
    test_df = data.iloc[val_end:]

    return train_df, val_df, test_df

train_24, val_24, test_24 = chronological_split(
    df_24,
    "target_P_dusty_24h"
)

print("Train:", train_24.shape)
print("Validation:", val_24.shape)
print("Test:", test_24.shape)

print("\nTrain period:", train_24.index.min(), "→", train_24.index.max())
print("Validation period:", val_24.index.min(), "→", val_24.index.max())
print("Test period:", test_24.index.min(), "→", test_24.index.max())

Train: (2990, 42)
Validation: (641, 42)
Test: (641, 42)

Train period: 2022-02-20 00:00:00 → 2022-06-24 13:00:00
Validation period: 2022-06-24 14:00:00 → 2022-07-21 06:00:00
Test period: 2022-07-21 07:00:00 → 2022-08-16 23:00:00


In [19]:
# Save processed datasets

df_24.to_parquet(OUTPUT_DIR / "forecast_24h.parquet")
df_48.to_parquet(OUTPUT_DIR / "forecast_48h.parquet")

with open(OUTPUT_DIR / "features.txt", "w") as f:
    f.write("\n".join(features))

print("Saved files:")
for file in OUTPUT_DIR.iterdir():
    print("-", file)

Saved files:
- processed_data\features.txt
- processed_data\forecast_24h.parquet
- processed_data\forecast_48h.parquet


In [ ]:
# !pip install pyarrow


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# Leakage sanity check

check_cols = [
    "P_dusty_lag_24h",
    "P_dusty_roll_mean_24h",
    "dust_lag_24h",
    "dust_roll_mean_24h"
]

for col in check_cols:
    original = df[target]
    shifted = df[col].shift(1)

    print(f"{col}: first non-null index = {df[col].first_valid_index()}")

P_dusty_lag_24h: first non-null index = 2022-02-18 00:00:00
P_dusty_roll_mean_24h: first non-null index = 2022-02-18 00:00:00
dust_lag_24h: first non-null index = 2022-02-18 00:00:00
dust_roll_mean_24h: first non-null index = 2022-02-18 00:00:00
